In [ ]:
#-- Packages --#

#--- Operational ---#
import os
import sys 
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path
import json
import re
import geopandas as gpd

#--- Visualisations ---#
import plotly.express as px
import plotly.graph_objects as go

#-- Directories --#
nb_dir = Path.cwd()
REPO_ROOT = nb_dir.parent

data_dir = REPO_ROOT / 'data/'
docs_dir = REPO_ROOT / 'docs/'

analysis_dir = data_dir / 'processed' / 'analysis/'
fires_dir = analysis_dir / "national_canadian_fires"

cache_dir = data_dir / 'processed' / 'cached/'

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [2]:
#-- Helper Functions --#
def extract_max_year(path):
    """
    Identify shapefile with latest year of available data
    """
    years = re.findall(r"\d{4}", path.stem)
    return max(map(int, years)) if years else -1

In [ ]:
#-- Load Files --#

# National Canada polygons (GeoJSON)
print(f'Loading National fires GeoJSON...')
fires_GeoJSON_path = analysis_dir / "national_canadian_fires/Canada_fires_1990_2024.geojson"
fires_GeoJSON = gpd.read_file(fires_GeoJSON_path)
print(f" National Canada fires GeoJSON loaded. {fires_GeoJSON.crs}\n")

# National Canada polygons (Shapefile)

shp_files = list(fires_dir.glob("*.shp"))

if not shp_files:
    raise FileNotFoundError(f"No shapefiles found in {fires_dir}\n")

fires_path = max(shp_files, key=extract_max_year)

print(f"Loading National fires shapefile... \n File name: {fires_path.name}")
fire_stats = gpd.read_file(fires_path)
print(f" National Canada Fires loaded. {fire_stats.crs}\n")


Loading National fires GeoJSON...
 National Canada fires GeoJSON loaded. EPSG:4326

Loading National fires shapefile... 
 File name: Canada_fires_1990_2024.shp
 National Canada Fires loaded. EPSG:4326



In [6]:
print(f"Cache National fires shapefile as parquet")
cached_path = cache_dir / f"{fires_path.name[:-4]}.parquet"
try:
    print(" Caching...")
    fire_stats.to_parquet(cached_path, index=False)
    print(f" Cache complete. \n Saved to: {cached_path}" )
except Exception as e:
    raise RuntimeError(f"National fires shapefile failed to export as parquet: {e}")

Cache National fires shapefile as parquet
 Caching...
 Cache complete. 
 Saved to: /Users/mitchellpalmer/Projects/canada-wildfire-avcan/data/processed/cached/Canada_fires_1990_2024.parquet


In [7]:
NBAC_stats = gpd.read_parquet(cached_path)


In [8]:
NBAC_stats.head(5)

,gid,fireid,year,prov_terr,natpark,adj_ha,cause,geometry
0,2007_1,1,2007,PC,WB,96802.352864,Natural,"MULTIPOLYGON (((-112.92482 59.07631, -112.9243..."
1,2007_2,2,2007,NT,None,20127.908056,Natural,"MULTIPOLYGON (((-105.30821 60.16116, -105.3093..."
2,2007_2,2,2007,SK,None,6615.977426,Natural,"MULTIPOLYGON (((-105.96553 60.00004, -105.9655..."
3,2007_3,3,2007,MB,None,16722.340460,Natural,"MULTIPOLYGON (((-97.05838 58.82598, -97.05781 ..."
4,2007_4,4,2007,MB,None,16593.539491,Natural,"MULTIPOLYGON (((-101.04291 58.27062, -101.0427..."


In [15]:
NBAC_stats.shape

(39924, 8)

In [10]:
year_totals = (
    fire_stats.groupby("year")
    .agg(
        fire_count=("gid", "nunique"),
        total_adj_ha=("adj_ha", "sum"),
    )
)

# De-duplicate to one row per fire per year (keeps its cause)
fires_unique = fire_stats.drop_duplicates(["year", "gid"])

cause_counts = (
    fires_unique.groupby(["year", "cause"])
    .size()
    .unstack(fill_value=0)
)

fire_years = year_totals.join(cause_counts)

# Percentages using total unique fires as denominator
cause_pct = cause_counts.div(fire_years["fire_count"], axis=0).add_suffix("_pct")*100
fire_years = fire_years.join(cause_pct)

fire_years.head()


,fire_count,total_adj_ha,Human,Natural,Undetermined,Human_pct,Natural_pct,Undetermined_pct
year,,,,,,,,
1990,509,8.587488e+05,94,226,189,18.467583,44.400786,37.131631
1991,575,1.530291e+06,150,201,224,26.086957,34.956522,38.956522
1992,331,8.640520e+05,82,136,113,24.773414,41.087613,34.138973
1993,377,1.947867e+06,56,163,158,14.854111,43.236074,41.909814
1994,708,5.073875e+06,78,323,307,11.016949,45.621469,43.361582


In [11]:
fire_means = fire_years[["fire_count", "total_adj_ha","Human_pct","Natural_pct","Undetermined_pct"]].mean()
fire_means

fire_count          1.131886e+03
total_adj_ha        2.584733e+06
Human_pct           2.564489e+01
Natural_pct         5.131808e+01
Undetermined_pct    2.303703e+01
dtype: float64

In [12]:
fire_means.astype(int)

fire_count             1131
total_adj_ha        2584733
Human_pct                25
Natural_pct              51
Undetermined_pct         23
dtype: int64

In [13]:
fire_means.astype('float').round(2)

fire_count             1131.89
total_adj_ha        2584733.31
Human_pct                25.64
Natural_pct              51.32
Undetermined_pct         23.04
dtype: float64

In [14]:
fire_total = fire_stats["fireid"].count()
